# B2.17 · Regression suites, cost curves and the economics of autonomy

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.16 · Performance testing: reliability under non-determinism](https://spbreed.github.io/cyber-commons/lessons/B2.16.html)**.

| | |
|---|---|
| Open-source tooling | Inspect, CyberGym |
| Open-weight models | GLM-5.2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A harness has three costs and most teams measure one of them.

**Accuracy** is measured — usually against whatever the tool happened to find,
which is circular. The fix is a **seeded corpus**: known defects planted
deliberately, so recall is a fraction with a real denominator instead of a
count with a confident tone.

**Money** is rarely measured per unit of value. Total spend is visible on an
invoice; *cost per confirmed finding* is the number that says whether the thing
is worth running, and it is the one that moves when someone changes the model
tier.

**Human time** is almost never measured, and it is the one that decides whether
the tool survives. A harness that produces four hundred findings a week has not
removed work if an analyst spends nine minutes on each. It has moved the work,
renamed it triage, and made it somebody else's problem.

The fourth number closes the loop: **regression drift.** A fixed suite re-run on
every harness change, because the most common failure mode in this whole
chapter is a change that improves the case you were looking at and quietly
degrades six you were not.

## 2 · A corpus with a real denominator

In [ ]:
SEEDED_CORPUS = {
 f"unit_{i:02d}": (i % 4 == 0, ["CWE-22","CWE-78","CWE-89","CWE-79"][i % 4])
 for i in range(40)
}
planted = sorted(u for u, (is_bug, _) in SEEDED_CORPUS.items() if is_bug)
print(f"units in corpus  : {len(SEEDED_CORPUS)}")
print(f"planted defects  : {len(planted)}")
print(f"denominator for recall is now a fact, not an estimate")

## 3 · Score a harness against it

In [ ]:
def harness_run(corpus, sensitivity, seed=2):
    """Higher sensitivity finds more real defects and more false ones."""
    import random
    rng = random.Random(seed)
    out = []
    for unit, (is_bug, cwe) in sorted(corpus.items()):
        if is_bug and rng.random() < sensitivity:
            out.append((unit, cwe, True))
        elif not is_bug and rng.random() < sensitivity * 0.35:
            out.append((unit, cwe, False))
    return out

def scorecard(found, corpus, tokens_per_unit=1800, usd_per_1k=0.002,
              analyst_minutes=6):
    tp = [f for f in found if f[2]]
    fp = [f for f in found if not f[2]]
    planted_n = sum(1 for _, (b, _) in corpus.items() if b)
    spend = len(corpus) * tokens_per_unit / 1000 * usd_per_1k
    return {
      "recall": len(tp) / planted_n,
      "precision": len(tp) / len(found) if found else 0.0,
      "true_positives": len(tp), "false_positives": len(fp),
      "spend_usd": round(spend, 3),
      "cost_per_finding_usd": round(spend / len(tp), 3) if tp else None,
      "analyst_minutes": len(found) * analyst_minutes,
      "minutes_per_accepted": round(len(found) * analyst_minutes / len(tp), 1) if tp else None,
    }

print(f"{'sens':>6}{'recall':>9}{'prec':>7}{'$/find':>9}{'min/accepted':>14}")
for s in (0.4, 0.7, 0.95):
    card = scorecard(harness_run(SEEDED_CORPUS, s), SEEDED_CORPUS)
    print(f"{s:>6.2f}{card['recall']:>9.0%}{card['precision']:>7.0%}"
          f"{card['cost_per_finding_usd']:>9.3f}{card['minutes_per_accepted']:>14}")

## 4 · Where it breaks — the tool that moved the work

In [ ]:
BASELINE_MANUAL_MINUTES = 40 * 6      # a human reviewing the same corpus

for s in (0.4, 0.7, 0.95):
    card = scorecard(harness_run(SEEDED_CORPUS, s), SEEDED_CORPUS)
    saved = BASELINE_MANUAL_MINUTES - card["analyst_minutes"]
    verdict = "saves time" if saved > 0 else "COSTS MORE TIME THAN IT SAVES"
    print(f"sensitivity {s:.2f}: analyst {card['analyst_minutes']:>4} min vs "
          f"manual {BASELINE_MANUAL_MINUTES} min -> {verdict}")
print()
print("At high sensitivity recall looks excellent and the review queue is")
print("longer than reading the code by hand. The accuracy metric improved and")
print("the thing got worse - which is invisible unless review load is tracked")
print("as a first-class number beside it.")

## 5 · The control — regression drift on every change

In [ ]:
FIXED_SUITE = dict(list(SEEDED_CORPUS.items())[:20])

def suite_result(sensitivity):
    found = harness_run(FIXED_SUITE, sensitivity, seed=9)
    return {u for u, _, is_tp in found if is_tp}

before = suite_result(0.7)
after_a = suite_result(0.75)      # the "improvement" someone shipped
after_b = suite_result(0.5)       # a change that helped one case

for label, after in (("tuned up", after_a), ("tuned down", after_b)):
    gained = sorted(after - before)
    lost = sorted(before - after)
    print(f"{label:12s} gained {len(gained):>2}  lost {len(lost):>2}  "
          f"net {len(after) - len(before):+d}")
    if lost:
        print(f"   regressions: {lost}")
print()
print("A net-positive change can still lose findings it used to catch. Net is")
print("the number people quote; the lost list is the number that pages someone")
print("three months later.")
assert (before - after_b)

## 6 · Verify — the full scorecard, published or it did not happen

In [ ]:
card = scorecard(harness_run(SEEDED_CORPUS, 0.7), SEEDED_CORPUS)
card.update({
  "corpus_size": len(SEEDED_CORPUS),
  "planted_defects": len(planted),
  "regression_suite": len(FIXED_SUITE),
  "regressions_last_change": len(before - after_b),
  "seed": 2,
})
for k in sorted(card):
    print(f"   {k:24s}{card[k]}")
print()
print("Ten fields, four of them about cost and human time. A harness reported")
print("on accuracy alone has answered the easiest question and skipped the two")
print("that decide whether anyone still runs it next quarter.")
assert card["cost_per_finding_usd"] and card["minutes_per_accepted"]

## What you just proved

A forty-unit corpus with ten planted defects gives recall a real denominator. Raising sensitivity improves recall while pushing the analyst review queue past the cost of reading the code by hand — the accuracy metric improves as the tool gets worse. A regression suite then shows a net-positive change that still lost findings it used to catch.

## Your turn

Compute cost per confirmed finding and analyst minutes per accepted finding for one tool you already run. If nobody has the second number, the tool's value is currently an article of faith.

---

**Next → [C1.1 · Agentic offensive workflow](https://spbreed.github.io/cyber-commons/lessons/C1.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.17.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.17.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*